# Subtracting a constant that changes everything

**Topic:** REINFORCE baselines &nbsp;·&nbsp; **Difficulty:** Medium

The policy gradient is `E[G * grad log pi]`. Subtracting any baseline `b` that
does not depend on the action leaves the expectation unchanged: the correction
term is `b * E[grad log pi]`, and that expectation is exactly zero. So you get to
subtract whatever you like, for free, and the variance can fall by several
orders of magnitude.

Verify the unbiasedness numerically, then find the variance-minimizing baseline
and compare it against the obvious choice.

## The data (created in the next cell)

- A 1-step bandit with 4 actions, softmax policy over fixed logits.
- Rewards have a large common offset, which is the situation baselines exist for.
- 20,000 sampled gradient estimates per baseline, `np.random.default_rng(700)`.

## Your task

With **NumPy only**:

1. For baselines `[0, mean_reward, optimal]`, estimate the policy gradient with
   respect to the logits by REINFORCE: sample an action, form
   `(G - b) * grad log pi(a)`.
2. The exact gradient is computable in closed form for this bandit. Compute it.
3. The variance-minimizing baseline for the scalar case is
   `E[G * ||grad log pi||^2] / E[||grad log pi||^2]`. Estimate it from the same
   20,000 draws.
4. Set `answers` to a dict with `"exact_grad"` `(4,)`, `"bias"` (3 values, the
   L2 distance from each estimator's mean to the exact gradient),
   `"variance"` (3 values, the mean per-coordinate variance), `"optimal_b"`, and
   `"var_reduction"`, the variance at `b = 0` divided by variance at the optimum.

## Evaluation

Compared to the reference to a relative tolerance of 1e-5, using two scalars
derived from each array (its sum and a fixed random projection) rather than every
element. The evaluation cell checks all three estimators are unbiased and that
both baselines cut variance substantially.

In [ ]:
import numpy as np

logits = np.array([0.4, -0.2, 0.9, 0.1])
rewards = np.array([100.3, 100.0, 101.2, 99.6])     # large common offset
n_samples = 20000
print(f"4-armed bandit, rewards {rewards} (offset ~100)")

In [ ]:
# =====================================================================
# YOUR SOLUTION GOES HERE
# =====================================================================
# Use the training data created above to build your model, then create
# the variable(s) the problem statement asks for (see "Your task").
# When you are done, run the evaluation cell below to check your work.

In [ ]:
# =====================================================================
# EVALUATION: run this cell to score your answer
# =====================================================================
import numpy as np

if "answers" not in dir():
    raise NameError("Define `answers` in the solution cell before running this cell.")
if not isinstance(answers, dict):
    raise TypeError("`answers` must be a dict.")

def _digest(a):
    a = np.asarray(a, dtype=float).ravel()
    if a.size == 0:
        return 0.0
    probe = np.random.default_rng(9176).standard_normal(a.size)
    return float(a @ probe)

_CHECKS = [
    ("exact_grad", [4], 7.105427357601002e-15, -0.712368164107357, 1e-05),
    ("bias", [3], 0.32827475142942314, -0.05569023692760725, 1e-05),
    ("variance", [3], 1775.2539113218324, -297.27509615832383, 1e-05),
    ("optimal_b", [], 100.31216302166766, -16.796069438096445, 1e-05),
    ("var_reduction", [], 54279.585030700626, -9088.465962495318, 1e-05),
]

print("=" * 56)
_ok = True
for _key, _shape, _sum, _probe, _tol in _CHECKS:
    if _key not in answers:
        print(f"  {_key:<24} MISSING")
        _ok = False
        continue
    _a = np.asarray(answers[_key], dtype=float)
    if list(_a.shape) != _shape:
        print(f"  {_key:<24} shape {list(_a.shape)}, expected {_shape}")
        _ok = False
        continue
    if not np.all(np.isfinite(_a)):
        print(f"  {_key:<24} contains NaN or inf")
        _ok = False
        continue
    _g = (abs(_a.sum() - _sum) <= _tol * (1.0 + abs(_sum))
          and abs(_digest(_a) - _probe) <= _tol * (1.0 + abs(_probe)))
    print(f"  {_key:<24} {'ok' if _g else 'wrong'}")
    _ok = _ok and _g

if "variance" in answers and "bias" in answers:
    _b = np.asarray(answers["bias"], dtype=float)
    _v = np.asarray(answers["variance"], dtype=float)
    _unbiased = float(_b.max()) < 0.5
    _cut = _v[2] * 20 < _v[0] and _v[1] * 20 < _v[0]
    print(f"  {'all three unbiased':<24} {'ok' if _unbiased else 'wrong'}  "
          f"(max bias {_b.max():.4f})")
    print(f"  {'baselines cut variance':<24} {'ok' if _cut else 'wrong'}  "
          f"({_v[0]:.1f} -> {_v[1]:.3f} (mean) -> {_v[2]:.3f} (optimal))")
    _ok = _ok and _unbiased and _cut

print("=" * 56)
if _ok:
    print("PASSED: nice work!")
else:
    print("NOT PASSED yet: revisit your solution and try again.")

## Solution

`E[grad log pi] = 0` for any policy, so subtracting a constant changes nothing in
expectation. That is the theorem, and the measured biases are within sampling
noise of zero (the per-coordinate standard error is about 0.3 at b = 0). What it
does change is the *scale* of the multiplier: with rewards near 100 the raw
estimator has an enormous common factor riding on the score, and removing it
leaves only the differences that carry information. That is why the value function
in actor-critic exists: it is a state-dependent baseline, and the advantage
`A = Q - V` is precisely the reward with the useless part subtracted off.

```python
import numpy as np

def softmax(z):
    z = z - z.max()
    e = np.exp(z)
    return e / e.sum()

p = softmax(logits)
K = len(logits)

grad_log = np.eye(K) - p[None, :]          # row a = grad log pi(a)
exact_grad = (p[:, None] * rewards[:, None] * grad_log).sum(axis=0)

rs = np.random.default_rng(700)
acts = rs.choice(K, size=n_samples, p=p)
G = rewards[acts]
GL = grad_log[acts]

sq = (GL ** 2).sum(axis=1)
optimal_b = float((G * sq).mean() / sq.mean())

baselines = [0.0, float((p * rewards).sum()), optimal_b]
bias, variance = [], []
for b in baselines:
    est = (G - b)[:, None] * GL
    bias.append(float(np.linalg.norm(est.mean(axis=0) - exact_grad)))
    variance.append(float(est.var(axis=0).mean()))

answers = {
    "exact_grad": exact_grad,
    "bias": np.array(bias),
    "variance": np.array(variance),
    "optimal_b": optimal_b,
    "var_reduction": float(variance[0] / variance[2]),
}
```